In [1]:
%load_ext cuml.accel
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import DBSCAN
from sklearn.metrics import classification_report, roc_auc_score

[2025-06-14 10:06:09.092] [CUML] [info] cuML: Installed accelerator for sklearn.
[2025-06-14 10:06:35.570] [CUML] [info] cuML: Installed accelerator for umap.
[2025-06-14 10:06:35.667] [CUML] [info] cuML: Installed accelerator for hdbscan.
[2025-06-14 10:06:35.667] [CUML] [info] cuML: Successfully initialized accelerator.


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kartik2112/fraud-detection")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/fraud-detection


In [3]:
df_train = pd.read_csv(path + "/fraudTrain.csv")
df_test = pd.read_csv(path + "/fraudTest.csv")

In [4]:
numerical_cols = ['amt', 'city_pop', 'unix_time', 'distance', 'hour', 'day_of_week', 'month']
categorical_cols = ['category']

In [5]:
def add_features(df):
    df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
    df['hour'] = df['trans_date_trans_time'].dt.hour
    df['day_of_week'] = df['trans_date_trans_time'].dt.dayofweek
    df['month'] = df['trans_date_trans_time'].dt.month

    def haversine(lat1, lon1, lat2, lon2):
        R = 6371
        lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
        c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
        return R * c

    df['distance'] = haversine(df['lat'], df['long'], df['merch_lat'], df['merch_long'])
    return df

df_train = add_features(df_train)
df_test = add_features(df_test)


In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_cols)
    ])

In [7]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('cluster', DBSCAN(eps=0.5, min_samples=5))
])

In [ ]:
X_train = df_train[numerical_cols + categorical_cols]
pipeline.fit(X_train)

In [ ]:
df_train['cluster'] = pipeline.predict(X_train)
df_train['is_anomaly'] = df_train['cluster'] == -1


In [ ]:
X_test = df_test[numerical_cols + categorical_cols]
df_test['cluster'] = pipeline.predict(X_test)
df_test['is_anomaly'] = df_test['cluster'] == -1

In [ ]:
print("Sample of test set clustering results:")
print(df_test[['amt', 'hour', 'category', 'cluster', 'is_anomaly', 'is_fraud']].head(10))
print("\nNumber of anomalies detected in test set:", df_test['is_anomaly'].sum())
print("Number of actual fraud cases in test set:", df_test['is_fraud'].sum())

In [ ]:
print("\nClassification Report (Anomaly as Fraud Prediction on Test Set):")
print(classification_report(df_test['is_fraud'], df_test['is_anomaly'], target_names=['Not Fraud', 'Fraud']))
print("ROC-AUC Score on Test Set:", roc_auc_score(df_test['is_fraud'], df_test['is_anomaly']))